# 5.5 What is the Fourier transform doing?

:::{margin}
The intuition presented here was largely inspired by [this excellent 3Blue1Brown video](https://www.youtube.com/watch?v=spUNpyF58BY).
:::

The definition of the Fourier transform above can feel like it appears out of nowhere. That is okay. Let us unpack it more intuitively.

How can a single integral possibly pick out the amount of one specific frequency $\omega$ hiding inside an arbitrary signal? Look again at the transform, $X(\omega) = \int x(t)\, e^{-j\omega t}\, dt$, and read it in three steps:

1. **Synthesize a phasor at $-\omega$.** The term $e^{-j\omega t}$ is a complex sinusoid rotating at frequency $\omega$, just in the clockwise direction (the minus sign reverses the direction of rotation).
1. **Multiply to measure similarity.** Multiplying the signal $x(t)$ by this phasor "winds" the signal around the complex plane at rate $\omega$. Wherever the signal's own oscillation matches the winding rate, the product reinforces in a consistent direction.
1. **Integrate to sum over time.** The integral adds up the wound signal across all time, accumulating that reinforcement (or lack of it) into a single complex number.

Intuitively, we are measuring the _correlation_ between $x(t)$ and a phasor probing at frequency $\omega$. The cleanest way to see this is to look at the wound-up signal in the complex plane and track its **center of mass** (the average of all the wound points). When the probe frequency matches a frequency present in the signal, the winding lines up and the center of mass is pulled far from the origin, yielding a large $|X(\omega)|$. When the probe frequency does not match, the winding smears symmetrically around the origin, the contributions cancel, and the center of mass sits near zero:

:::{figure}
![Three complex-plane plots of the same signal wound at different probe frequencies. At a probe of 2 Hz the curve forms balanced lobes and the center of mass sits at the origin. At 3 Hz, matching the signal, the curve bunches to one side and the center of mass is pulled well away from the origin. At 4 Hz the center of mass returns to near the origin.](./assets/fig-ft-intuition.png)

Winding a signal (here a 3 Hz oscillation) around the complex plane at three probe frequencies. The red dot is the center of mass. Only at the matching probe of 3 Hz (middle) is the center of mass pulled away from the origin, signaling a large amplitude at that frequency. At non-matching probes (2 and 4 Hz), the contributions cancel and the center of mass stays near zero.
:::

The center of mass is an average, and the integral in the Fourier transform is a (continuous) sum, so the two are proportional. The Fourier transform sweeps this probe across every frequency $\omega$ and records, for each one, how far off-origin the center of mass lands.

In [ ]:
# hide
# --- Dependencies -----------------------------------------------------------
# All imports and installs for this page live here. The installs are no-ops
# when the packages are already present (they ship with the Jupyter Book build
# environment), so this cell just needs to run once. The capture_output()
# block hides pip's chatter but still lets a genuine install failure raise.
# myst-nb isn't installed here — it's build-time only, and the browser kernel
# stubs its glue() to a no-op.
from IPython.utils.capture import capture_output
with capture_output():                 # swallow pip + import chatter (e.g. matplotlib's font cache); real failures still raise
    # numpy + matplotlib are Pyodide built-ins; the install is a no-op at build time
    %pip install -q numpy matplotlib
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib import gridspec
    from matplotlib.animation import FuncAnimation
    from myst_nb import glue
    from IPython.display import HTML


In [ ]:
# hide
plt.rcParams.update({
    "figure.dpi": 64,                # sets the player's intrinsic size only —
    "figure.subplot.bottom": 0.18,   # reserve room for x-axis labels (jshtml crops at the figure bbox)
    "animation.html": "jshtml",      # SVG frames are vector, sharp at any zoom
    "animation.frame_format": "svg", # ~3x smaller gzipped than PNG, and crisp
    "animation.embed_limit": 40,     # MB; matplotlib's 20 MB default drops frames
    "text.usetex": False,            # mathtext: no LaTeX install needed locally
    "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#6D6E71",
    "xtick.color": "#6D6E71", "ytick.color": "#6D6E71",
    "xtick.labelcolor": "#3b3b3b", "ytick.labelcolor": "#3b3b3b",
    "axes.labelcolor": "#3b3b3b",
})

# House palette (interactive template, "The house style"): color is meaning.
C2 = "#007BC0"   # Highlands blue — the 2 Hz component
C4 = "#008F91"   # teal thread   — the 4 Hz component
CT = "#C41230"   # Carnegie red  — the measured thing: the center of mass
CP = "#FDB515"   # gold thread   — the moving part: the probe
CX = "#6D6E71"   # iron gray     — the composite signal x(t)
CG = "#E0E0E0"   # steel gray    — static guides: unit circle, axes lines

# --- A two-tone signal: actual sines, accurate amplitudes (more 2 Hz) ------
T, n_pts = 2.0, 400
t = np.linspace(0.0, T, n_pts, endpoint=False)
A2, F2, A4, F4 = 1.0, 2.0, 0.5, 4.0
x2 = A2 * np.sin(2 * np.pi * F2 * t)
x4 = A4 * np.sin(2 * np.pi * F4 * t)
x = x2 + x4

# --- Sweep the probe frequency; wound components and centers of mass -------
frame_rate, n_frames = 24, 120
g = np.linspace(0.5, 6.0, n_frames)
ph = np.exp(-1j * 2 * np.pi * np.outer(g, t))
w2, w4 = x2[None, :] * ph, x4[None, :] * ph
com2, com4 = w2.mean(1), w4.mean(1)
com = com2 + com4
mag, mag2, mag4 = np.abs(com), np.abs(com2), np.abs(com4)
ymax = mag.max() * 1.3

# Tone frequencies and their spectrum heights, for the "discovery" stars.
peaks = []
for f, c in [(F2, C2), (F4, C4)]:
    k = int(np.argmin(np.abs(g - f)))
    lo, hi = max(0, k - 5), min(n_frames, k + 6)
    kk = lo + int(np.argmax(mag[lo:hi]))
    peaks.append((g[kk], mag[kk], c, f"{f:g} Hz"))

# --- Layout: winding (big, left) | signal (top-right) / spectrum (bottom) --
fig = plt.figure(figsize=(10.5, 5.6))
gs = gridspec.GridSpec(2, 2, width_ratios=[1.25, 1], height_ratios=[1, 1.55],
                       hspace=0.55, wspace=0.3)
ax_w = fig.add_subplot(gs[:, 0])
ax_sig = fig.add_subplot(gs[0, 1])
ax_s = fig.add_subplot(gs[1, 1])

# signal panel: color-coded tones (the equation) plus the animated dashed PROBE,
# whose wavelength shrinks as the probe frequency rises (the 3Blue1Brown effect).
ax_sig.plot(t, x, color=CX, lw=2.2, label=r"$x(t)$", zorder=3)
ax_sig.plot(t, x2, color=C2, lw=1.5, alpha=0.75, label=r"$\sin(2\pi\,2\,t)$", zorder=1)
ax_sig.plot(t, x4, color=C4, lw=1.5, alpha=0.75, label=r"$\frac{1}{2}\sin(2\pi\,4\,t)$", zorder=1)
probe_line, = ax_sig.plot(t, np.sin(2 * np.pi * g[0] * t), color=CP, lw=1.9,
                          ls="--", alpha=0.95, label=r"probe $\sin(2\pi g\,t)$", zorder=4)
ax_sig.set_xlim(0, T); ax_sig.set_ylim(-1.8, 2.3); ax_sig.set_yticks([])
ax_sig.set_xlabel("time (s)")
ax_sig.legend(loc="upper center", fontsize=8.5, labelcolor="linecolor", ncol=2,
              handlelength=1.3, columnspacing=1.0, borderpad=0.3,
              bbox_to_anchor=(0.5, 1.34)).get_frame().set_alpha(0.0)

theta = np.linspace(0, 2 * np.pi, 220)


def _arrow(p0, p1, color):
    ax_w.annotate("", xy=(p1.real, p1.imag), xytext=(p0.real, p0.imag),
                  arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2,
                                  shrinkA=0, shrinkB=0))


def draw_winding(n):
    ax_w.clear()
    ax_w.set_xlim(-1.18, 1.18); ax_w.set_ylim(-1.18, 1.18)
    ax_w.set_aspect("equal", "box")
    ax_w.plot(np.cos(theta), np.sin(theta), color=CG, lw=1.2, zorder=0)
    ax_w.axhline(0, color=CG, lw=1, zorder=0)
    ax_w.axvline(0, color=CG, lw=1, zorder=0)
    ax_w.plot(w2[n].real, w2[n].imag, color=C2, lw=1.6, alpha=0.55, zorder=2)
    ax_w.plot(w4[n].real, w4[n].imag, color=C4, lw=1.6, alpha=0.55, zorder=2)
    c2, ct = com2[n], com[n]
    _arrow(0 + 0j, c2, C2)                  # 2 Hz contribution
    _arrow(c2, ct, C4)                       # 4 Hz contribution, tip-to-tail
    ax_w.plot(ct.real, ct.imag, "o", color=CT, ms=22, alpha=0.18, zorder=4)
    ax_w.plot(ct.real, ct.imag, "o", color=CT, ms=10, mec="white", mew=1.3, zorder=5)
    ax_w.set_xlabel("real"); ax_w.set_ylabel("imaginary")
    ax_w.set_title(rf"probe  $g$ = {g[n]:.2f} Hz", fontsize=12.5)


def draw_spectrum(n):
    ax_s.clear()
    gg = g[:n + 1]
    ax_s.fill_between(gg, mag[:n + 1], color=CX, alpha=0.08)
    ax_s.plot(gg, mag2[:n + 1], color=C2, lw=1.4, alpha=0.8)
    ax_s.plot(gg, mag4[:n + 1], color=C4, lw=1.4, alpha=0.8)
    ax_s.plot(gg, mag[:n + 1], color=CX, lw=2.2)
    ax_s.plot(g[n], mag[n], "o", color=CT, ms=8, mec="white", zorder=6)
    for gp, yp, c, label in peaks:
        if g[n] >= gp - 0.01:
            ax_s.plot(gp, yp, "*", color=c, ms=16, zorder=7)
            ax_s.annotate(label, (gp, yp), textcoords="offset points", xytext=(0, 9),
                          ha="center", color=c, fontsize=11, fontweight="bold")
    ax_s.set_xlim(g[0], g[-1]); ax_s.set_ylim(0, ymax)
    ax_s.set_xlabel(r"probe frequency  $g$  (Hz)")
    ax_s.set_ylabel("|center of mass|")


def animate(n):
    probe_line.set_ydata(np.sin(2 * np.pi * g[n] * t))   # probe period shrinks with g
    draw_winding(n)
    draw_spectrum(n)
    return ()


anim = FuncAnimation(fig, animate, frames=np.arange(n_frames),
                     interval=1000 / frame_rate, blit=False)
player = HTML(anim.to_jshtml())
plt.close(fig)

# Register for the book page (embed with `{glue}`winding-sweep``); the player
# below previews the animation inline here in the notebook.
glue("winding-sweep", player, display=False)
player
